In [0]:
spark.sql("USE CATALOG chat_bot")

for schema in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS chat_bot.{schema}")

spark.sql("SHOW SCHEMAS IN chat_bot").display()

In [0]:
products = [
    (1, "Laptop Pro 14", "Laptops", 1200.0, 45),
    (2, "Laptop Air 13", "Laptops", 950.0, 60),
    (3, "Gaming Laptop X", "Laptops", 1800.0, 20),
    (4, "Phone S10", "Phones", 800.0, 120),
    (5, "Phone S10 Mini", "Phones", 600.0, 90),
    (6, "Phone Ultra", "Phones", 1100.0, 70),
    (7, "Wireless Mouse", "Accessories", 25.0, 300),
    (8, "Mechanical Keyboard", "Accessories", 85.0, 150),
    (9, "USB-C Hub", "Accessories", 40.0, 200),
    (10, "Noise Cancelling Headphones", "Audio", 220.0, 80),
    (11, "Bluetooth Speaker", "Audio", 90.0, 110),
    (12, "Earbuds Lite", "Audio", 55.0, 250),
]

df=spark.createDataFrame(products,['product_id','product_name','category','price','stock'])
df.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('chat_bot.bronze.products')
spark.table('chat_bot.bronze.products').display()

In [0]:
import random
from datetime import date,timedelta

regions = ["North", "South", "East", "West"]
segments = ["Retail", "Corporate", "Online"]

customer=[

    (i,f"customer{i}",random.choice(regions),random.choice(segments)) for i in range(1,51)
]

spark.createDataFrame(customer,['customer_id','customer_name','region','segment']).write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('chat_bot.bronze.customers')

prices = {r["product_id"]: r["price"] for r in spark.table("chat_bot.bronze.products").collect()}

start=date(2026,4,1)
sales=[]

for sale_id in range(1, 801):
    pid = random.randint(1, 12)
    qty = random.randint(1, 5)
    sale_date = start + timedelta(days=random.randint(0, 180))
    sales.append((sale_id, sale_date, random.randint(1, 50), pid, qty, round(prices[pid] * qty, 2)))

spark.createDataFrame(
    sales, ["sale_id", "sale_date", "customer_id", "product_id", "quantity", "total_amount"]
).write.mode("overwrite").saveAsTable("chat_bot.bronze.sales")















In [0]:
sp = "94dca96b-a235-4095-84de-a6ad28a0ab74"

spark.sql(f"GRANT USE CATALOG ON CATALOG chat_bot TO `{sp}`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA chat_bot.gold TO `{sp}`")
spark.sql(f"GRANT SELECT ON SCHEMA chat_bot.gold TO `{sp}`")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
for wh in w.warehouses.list():
    print(wh.id, wh.name, wh.state)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS chat_bot.gold.chatbot_query_logs (
    request_id STRING,
    question STRING,
    sql_query STRING,
    created_at TIMESTAMP
)
USING DELTA;

In [0]:
%sql
select * from chat_bot.gold.chatbot_query_logs

In [0]:
%sql
CREATE TABLE IF NOT EXISTS chat_bot.gold.chatbot_feedback (
    request_id STRING,
    feedback STRING,
    created_at TIMESTAMP
)
USING DELTA;




In [0]:
%sql
GRANT ALL PRIVILEGES ON TABLE chat_bot.gold.chatbot_feedback TO `94dca96b-a235-4095-84de-a6ad28a0ab74`

In [0]:
%sql
select * from chat_bot.gold.chatbot_feedback

In [0]:
%sql
select * from chat_bot.gold.chatbot_query_logs